In [0]:
%run ../utils/api_client

In [0]:
%run ../utils/logger

In [0]:
%run ../utils/delta_writer

In [0]:
%run ../utils/metadata_manager

In [0]:
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text(
    "endpoint",
    ""
)

In [0]:
endpoint = dbutils.widgets.get("endpoint")

In [0]:
table_name = f"bronze_{endpoint}"

In [0]:
last_execution = get_last_execution(
    table_name
)

print(
    f"Última execução: {last_execution}"
)

In [0]:
try:

    log_info(
        f"Iniciando ingestão endpoint: {endpoint}"
    )

    params = build_incremental_params(
        endpoint=endpoint,
        last_execution=last_execution
    )    

    dados = get_all_pages(
        endpoint,
        params=params
    )

    # ==========================================
    # TRATAMENTO SEM NOVOS DADOS
    # ==========================================

    if len(dados) == 0:

        log_info(
            f"Nenhum dado novo encontrado para {endpoint}"
        )

        register_execution(
            table_name=table_name,
            endpoint=endpoint,
            status="SUCCESS",
            record_count=0,
            error_message=None
        )

        dbutils.notebook.exit(
            "Sem novos dados."
        )

    # ==========================================
    # DATAFRAME
    # ==========================================

    df = spark.createDataFrame(dados)

    df = (
        df
        .withColumn(
            "dt_ingestao",
            F.current_timestamp()
        )
    )

    display(df)

    # ==========================================
    # PATH BRONZE
    # ==========================================

    bronze_path = (
        f"/Volumes/workspace/default/"
        f"camara_deputados/bronze/{endpoint}"
    )

    # ==========================================
    # WRITE DELTA
    # ==========================================

    write_delta(
        df=df,
        path=bronze_path,
        mode="overwrite"
    )

    # ==========================================
    # REGISTRO EXECUÇÃO
    # ==========================================

    register_execution(
        table_name=table_name,
        endpoint=endpoint,
        status="SUCCESS",
        record_count=df.count(),
        error_message=None
    )

    log_info(
        f"Ingestão finalizada tabela: {table_name}"
    )

except Exception as e:

    register_execution(
        table_name=table_name,
        endpoint=endpoint,
        status="ERROR",
        record_count=0,
        error_message=str(e)
    )

    log_error(str(e))

    raise

In [0]:
%skip
spark.sql("""

SELECT *
FROM workspace.default.metadata_ingestion_control
ORDER BY execution_time DESC

""").display()

In [0]:
%skip
%sql
SELECT count(*)
FROM workspace.default.bronze_deputados

In [0]:
%sql
SELECT count(*)
FROM delta.`/Volumes/workspace/default/camara_deputados/bronze/orgaos`